In [2]:
!uv add torchvision

Resolved 136 packages in 1.79s                                       
⠼ Preparing packages... (0/1)                                                   
⠴ Preparing packages... (0/1)-------------------     0 B/1.77 MiB            
⠴ Preparing packages... (0/1)-------------------     0 B/1.77 MiB            
⠴ Preparing packages... (0/1)------------------- 16.00 KiB/1.77 MiB          
⠴ Preparing packages... (0/1)------------------- 32.00 KiB/1.77 MiB          
⠴ Preparing packages... (0/1)------------------- 48.00 KiB/1.77 MiB          
⠴ Preparing packages... (0/1)------------------- 64.00 KiB/1.77 MiB          
⠦ Preparing packages... (0/1)------------------- 80.00 KiB/1.77 MiB          
⠦ Preparing packages... (0/1)------------------- 80.00 KiB/1.77 MiB          
⠦ Preparing packages... (0/1)------------------- 96.00 KiB/1.77 MiB          
⠦ Preparing packages... (0/1)------------------- 112.00 KiB/1.77 MiB         
⠦ Preparing packages... (0/1)------------------- 128.00 KiB/1.77 MiB 

In [3]:
import torch
import torchvision 
from torchvision import transforms 
from torch.utils.data import DataLoader, TensorDataset


image_path = './data'
transform = transforms.Compose([transforms.ToTensor()])

mnist_train_dataset = torchvision.datasets.MNIST(root=image_path, 
                                           train=True, 
                                           transform=transform, 
                                           download=True)
mnist_test_dataset = torchvision.datasets.MNIST(root=image_path, 
                                           train=False, 
                                           transform=transform, 
                                           download=False)
 
batch_size = 64
torch.manual_seed(1)
train_dl = DataLoader(mnist_train_dataset, batch_size, shuffle=True)

In [5]:
import torch.nn as nn

In [6]:
hidden_units = [32, 16]
image_size = mnist_train_dataset[0][0].shape
input_size = image_size[0] * image_size[1] * image_size[2]

all_layers = [nn.Flatten()]
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 10))
model = nn.Sequential(*all_layers)

model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=32, bias=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Linear(in_features=16, out_features=10, bias=True)
)

In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.manual_seed(1)
num_epochs = 20
for epoch in range(num_epochs):
    accuracy_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
        accuracy_hist_train += is_correct.sum()
    accuracy_hist_train /= len(train_dl.dataset)
    print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}')

Epoch 0  Accuracy 0.8531
Epoch 1  Accuracy 0.9287
Epoch 2  Accuracy 0.9413
Epoch 3  Accuracy 0.9506
Epoch 4  Accuracy 0.9558
Epoch 5  Accuracy 0.9592
Epoch 6  Accuracy 0.9627
Epoch 7  Accuracy 0.9650
Epoch 8  Accuracy 0.9674
Epoch 9  Accuracy 0.9690
Epoch 10  Accuracy 0.9710
Epoch 11  Accuracy 0.9729
Epoch 12  Accuracy 0.9739
Epoch 13  Accuracy 0.9750
Epoch 14  Accuracy 0.9765
Epoch 15  Accuracy 0.9778
Epoch 16  Accuracy 0.9778
Epoch 17  Accuracy 0.9798
Epoch 18  Accuracy 0.9806
Epoch 19  Accuracy 0.9811


In [8]:
pred = model(mnist_test_dataset.data / 255.)
is_correct = (torch.argmax(pred, dim=1) == mnist_test_dataset.targets).float()
print(f'Test accuracy: {is_correct.mean():.4f}') 

Test accuracy: 0.9649
